In [47]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

PROJECT_ROOT = Path("/home/iailab42/khans1/projects/ir")

# CANDIDATE = "posterior_bank"
# CANDIDATE = "prior"
CANDIDATE = "timevae"

VALID_CANDIDATES = ["posterior_bank", "prior", "timevae"]

if CANDIDATE not in VALID_CANDIDATES:
    raise ValueError(f"Invalid candidate: {CANDIDATE}. Choose from {VALID_CANDIDATES}")

DATA_DIR = PROJECT_ROOT / "data"
SYN_DIR = DATA_DIR / "synthetic" / CANDIDATE

RESULTS_DIR = PROJECT_ROOT / "results" / CANDIDATE / "realism_diversity"
FIGURES_DIR = PROJECT_ROOT / "figures" / CANDIDATE / "realism_diversity"
MODELS_DIR = PROJECT_ROOT / "models" / CANDIDATE
CONFIG_DIR = PROJECT_ROOT / "configs" / CANDIDATE / "realism_diversity"
LOG_DIR = PROJECT_ROOT / "logs" / CANDIDATE
CSV_DIR = RESULTS_DIR / "csv"

# Compatibility alias if older code still uses PLOTS_DIR
PLOTS_DIR = FIGURES_DIR

for directory in [SYN_DIR, RESULTS_DIR, FIGURES_DIR, MODELS_DIR, CONFIG_DIR, LOG_DIR, CSV_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REAL_X_PATH = DATA_DIR / "all_X.npy"
REAL_Y_PATH = DATA_DIR / "all_y.npy"
REAL_SUBJECT_PATH = DATA_DIR / "all_subject.npy"
REAL_METADATA_PATH = DATA_DIR / "all_metadata.csv"

SYN_X_PATH = SYN_DIR / "all_X_synthetic.npy"
SYN_Y_PATH = SYN_DIR / "all_y_synthetic.npy"
SYN_SUBJECT_PATH = SYN_DIR / "all_subject_synthetic.npy"
SYN_METADATA_PATH = SYN_DIR / "all_metadata_synthetic.csv"

CHANNEL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]
SIGNAL_NAMES = CHANNEL_NAMES

ACTIVITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8]

TARGET_HZ = 64
WINDOW_LEN = 512
NUM_CHANNELS = 6

TRAIN_SUBJECTS = [
    "S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"
]
VAL_SUBJECTS = ["S14", "S15"]
TEST_SUBJECTS = ["S7", "S8", "S10"]

REALISM_REFERENCE_SPLIT = "train"

MAX_REAL_WINDOWS_PER_ACTIVITY_FOR_REALISM = 3000
MAX_SYN_WINDOWS_PER_ACTIVITY_FOR_REALISM = 3000
MAX_REAL_TRAIN_WINDOWS_PER_ACTIVITY_FOR_DISTANCE = 1000
MAX_SYN_WINDOWS_PER_ACTIVITY_FOR_DISTANCE = 1000

HISTOGRAM_BINS = 80
FFT_EPS = 1e-8
DISTANCE_BATCH_SIZE = 256
RANDOM_SEED = 42

RUN_CONFIG = {
    "project_root": str(PROJECT_ROOT),
    "candidate": CANDIDATE,
    "data_dir": str(DATA_DIR),
    "synthetic_dir": str(SYN_DIR),
    "results_dir": str(RESULTS_DIR),
    "figures_dir": str(FIGURES_DIR),
    "models_dir": str(MODELS_DIR),
    "config_dir": str(CONFIG_DIR),
    "log_dir": str(LOG_DIR),
    "real_x_path": str(REAL_X_PATH),
    "real_y_path": str(REAL_Y_PATH),
    "real_subject_path": str(REAL_SUBJECT_PATH),
    "real_metadata_path": str(REAL_METADATA_PATH),
    "synthetic_x_path": str(SYN_X_PATH),
    "synthetic_y_path": str(SYN_Y_PATH),
    "synthetic_subject_path": str(SYN_SUBJECT_PATH),
    "synthetic_metadata_path": str(SYN_METADATA_PATH),
}

with open(CONFIG_DIR / "run_config.json", "w") as file:
    json.dump(RUN_CONFIG, file, indent=4)

print("Project root:", PROJECT_ROOT)
print("Candidate:", CANDIDATE)
print("Real data:", DATA_DIR)
print("Synthetic data:", SYN_DIR)
print("Results:", RESULTS_DIR)
print("Figures:", FIGURES_DIR)
print("Configs:", CONFIG_DIR)

Project root: /home/iailab42/khans1/projects/ir
Candidate: timevae
Real data: /home/iailab42/khans1/projects/ir/data
Synthetic data: /home/iailab42/khans1/projects/ir/data/synthetic/timevae
Results: /home/iailab42/khans1/projects/ir/results/timevae/realism_diversity
Figures: /home/iailab42/khans1/projects/ir/figures/timevae/realism_diversity
Configs: /home/iailab42/khans1/projects/ir/configs/timevae/realism_diversity


In [48]:
# ============================================================
# File validation
# ============================================================

required_files = {
    "REAL_X": REAL_X_PATH,
    "REAL_Y": REAL_Y_PATH,
    "REAL_SUBJECT": REAL_SUBJECT_PATH,
    "SYN_X": SYN_X_PATH,
    "SYN_Y": SYN_Y_PATH,
    "SYN_SUBJECT": SYN_SUBJECT_PATH,
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

print("All required files found.")

All required files found.


In [49]:
# ============================================================
# Basic helpers
# ============================================================

def require_file(path: Path) -> Path:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    return path


def check_window_shape(X: np.ndarray, expected_channels: int, name: str) -> None:
    if X.ndim != 3:
        raise ValueError(f"{name} must be 3D [N, T, C], got shape {X.shape}")

    if X.shape[1] != WINDOW_LEN:
        raise ValueError(f"{name} expected window length {WINDOW_LEN}, got {X.shape[1]}")

    if X.shape[2] != expected_channels:
        raise ValueError(f"{name} expected {expected_channels} channels, got {X.shape[2]}")


def subject_sort_key(subject_name: str):
    subject_name = str(subject_name)

    if subject_name.startswith("S") and subject_name[1:].isdigit():
        return int(subject_name[1:])

    return subject_name


def safe_mean(values: np.ndarray) -> float:
    values = np.asarray(values)

    if values.size == 0:
        return np.nan

    return float(np.mean(values))


def safe_std(values: np.ndarray) -> float:
    values = np.asarray(values)

    if values.size == 0:
        return np.nan

    return float(np.std(values))


def safe_percentile(values: np.ndarray, percentile: float) -> float:
    values = np.asarray(values)

    if values.size == 0:
        return np.nan

    return float(np.percentile(values, percentile))


def sample_indices_for_activity(
    y: np.ndarray,
    activity_id: int,
    max_windows: int,
    rng: np.random.Generator,
) -> np.ndarray:
    indices = np.where(y == activity_id)[0]

    if len(indices) == 0:
        return indices

    if len(indices) > max_windows:
        indices = rng.choice(indices, size=max_windows, replace=False)

    return np.asarray(indices, dtype=np.int64)

In [50]:
# ============================================================
# Data loading and filtering
# ============================================================

def load_real_data():
    require_file(REAL_X_PATH)
    require_file(REAL_Y_PATH)
    require_file(REAL_SUBJECT_PATH)

    X = np.load(REAL_X_PATH).astype(np.float32)
    y = np.load(REAL_Y_PATH).astype(np.int64)
    subjects = np.load(REAL_SUBJECT_PATH, allow_pickle=True).astype(str)

    if REAL_METADATA_PATH.exists():
        metadata = pd.read_csv(REAL_METADATA_PATH)
    else:
        metadata = pd.DataFrame({"row_id": np.arange(len(y))})

    check_window_shape(X, NUM_CHANNELS, "real X")

    if len(X) != len(y) or len(X) != len(subjects):
        raise ValueError("Real X, y, and subject arrays do not have matching lengths.")

    if len(metadata) != len(y):
        raise ValueError("Real metadata row count does not match y length.")

    return X, y, subjects, metadata


def load_synthetic_data():
    require_file(SYN_X_PATH)
    require_file(SYN_Y_PATH)
    require_file(SYN_SUBJECT_PATH)

    X = np.load(SYN_X_PATH).astype(np.float32)
    y = np.load(SYN_Y_PATH).astype(np.int64)
    subjects = np.load(SYN_SUBJECT_PATH, allow_pickle=True).astype(str)

    if SYN_METADATA_PATH.exists():
        metadata = pd.read_csv(SYN_METADATA_PATH)
    else:
        metadata = pd.DataFrame({"row_id": np.arange(len(y))})

    check_window_shape(X, NUM_CHANNELS, "synthetic X")

    if len(X) != len(y) or len(X) != len(subjects):
        raise ValueError("Synthetic X, y, and subject arrays do not have matching lengths.")

    if len(metadata) != len(y):
        raise ValueError("Synthetic metadata row count does not match y length.")

    return X, y, subjects, metadata


def filter_by_subjects(X, y, subjects, metadata, selected_subjects):
    selected_subjects = set(selected_subjects)
    mask = np.array([subject in selected_subjects for subject in subjects], dtype=bool)

    return (
        X[mask],
        y[mask],
        subjects[mask],
        metadata.loc[mask].reset_index(drop=True),
    )


def get_signal_windows(X: np.ndarray, signal_name: str) -> np.ndarray:
    channel_index = CHANNEL_NAMES.index(signal_name)
    return X[:, :, channel_index]

In [51]:
# ============================================================
# Realism metrics
# ============================================================

def histogram_overlap(
    real_values: np.ndarray,
    synthetic_values: np.ndarray,
    bins: int = HISTOGRAM_BINS,
) -> float:
    real_values = np.asarray(real_values).reshape(-1)
    synthetic_values = np.asarray(synthetic_values).reshape(-1)

    if real_values.size == 0 or synthetic_values.size == 0:
        return np.nan

    combined = np.concatenate([real_values, synthetic_values])
    lower = float(np.min(combined))
    upper = float(np.max(combined))

    if np.isclose(lower, upper):
        return 1.0

    real_hist, bin_edges = np.histogram(
        real_values,
        bins=bins,
        range=(lower, upper),
        density=True,
    )

    synthetic_hist, _ = np.histogram(
        synthetic_values,
        bins=bin_edges,
        density=True,
    )

    bin_widths = np.diff(bin_edges)
    overlap = np.sum(np.minimum(real_hist, synthetic_hist) * bin_widths)

    return float(np.clip(overlap, 0.0, 1.0))


def average_log_fft(windows: np.ndarray) -> np.ndarray:
    if windows.size == 0:
        return np.array([], dtype=np.float32)

    fft_values = np.fft.rfft(windows, axis=1)
    magnitude = np.abs(fft_values)
    log_magnitude = np.log1p(magnitude + FFT_EPS)

    return log_magnitude.mean(axis=0).astype(np.float32)


def fft_logmag_mae(real_windows: np.ndarray, synthetic_windows: np.ndarray) -> float:
    real_fft = average_log_fft(real_windows)
    synthetic_fft = average_log_fft(synthetic_windows)

    if real_fft.size == 0 or synthetic_fft.size == 0:
        return np.nan

    min_len = min(len(real_fft), len(synthetic_fft))

    return float(np.mean(np.abs(real_fft[:min_len] - synthetic_fft[:min_len])))


def compute_realism_table(real_X, real_y, synthetic_X, synthetic_y, reference_name: str):
    rng = np.random.default_rng(RANDOM_SEED)
    rows = []

    for signal_name in SIGNAL_NAMES:
        real_signal = get_signal_windows(real_X, signal_name)
        synthetic_signal = get_signal_windows(synthetic_X, signal_name)

        print("\n" + "=" * 70)
        print(f"Realism metrics | Candidate: {CANDIDATE} | Signal: {signal_name}")
        print("=" * 70)

        for activity_id in ACTIVITY_IDS:
            real_indices = sample_indices_for_activity(
                y=real_y,
                activity_id=activity_id,
                max_windows=MAX_REAL_WINDOWS_PER_ACTIVITY_FOR_REALISM,
                rng=rng,
            )

            synthetic_indices = sample_indices_for_activity(
                y=synthetic_y,
                activity_id=activity_id,
                max_windows=MAX_SYN_WINDOWS_PER_ACTIVITY_FOR_REALISM,
                rng=rng,
            )

            real_sample = real_signal[real_indices]
            synthetic_sample = synthetic_signal[synthetic_indices]

            if len(real_sample) == 0 or len(synthetic_sample) == 0:
                continue

            real_mean = safe_mean(real_sample)
            synthetic_mean = safe_mean(synthetic_sample)

            real_std = safe_std(real_sample)
            synthetic_std = safe_std(synthetic_sample)

            row = {
                "candidate": CANDIDATE,
                "activity": int(activity_id),
                "signal": signal_name,
                "reference_split": reference_name,
                "real_sampled_windows": int(len(real_sample)),
                "synthetic_sampled_windows": int(len(synthetic_sample)),
                "real_mean": real_mean,
                "synthetic_mean": synthetic_mean,
                "abs_mean_diff_lower_is_better": abs(real_mean - synthetic_mean),
                "real_std": real_std,
                "synthetic_std": synthetic_std,
                "abs_std_diff_lower_is_better": abs(real_std - synthetic_std),
                "histogram_overlap_0_to_1_higher_is_better": histogram_overlap(
                    real_sample,
                    synthetic_sample,
                ),
                "fft_logmag_mae_lower_is_better": fft_logmag_mae(
                    real_sample,
                    synthetic_sample,
                ),
            }

            rows.append(row)

    df = pd.DataFrame(rows)
    out_path = CSV_DIR / "realism_by_activity_signal.csv"
    df.to_csv(out_path, index=False)
    print("\nSaved:", out_path)

    return df


def save_realism_summary_tables(realism_df):
    metric_cols = [
        "abs_mean_diff_lower_is_better",
        "abs_std_diff_lower_is_better",
        "histogram_overlap_0_to_1_higher_is_better",
        "fft_logmag_mae_lower_is_better",
    ]

    avg_by_signal = (
        realism_df
        .groupby("signal", as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    count_by_signal = (
        realism_df
        .groupby("signal", as_index=False)[
            ["real_sampled_windows", "synthetic_sampled_windows"]
        ]
        .sum()
    )

    avg_by_signal = avg_by_signal.merge(count_by_signal, on="signal", how="left")
    avg_by_signal.insert(0, "candidate", CANDIDATE)

    avg_by_signal = avg_by_signal[
        ["candidate", "signal", "real_sampled_windows", "synthetic_sampled_windows"] + metric_cols
    ]

    out_path = CSV_DIR / "realism_average_by_signal.csv"
    avg_by_signal.to_csv(out_path, index=False)
    print("Saved:", out_path)

    overall = realism_df[metric_cols].mean(numeric_only=True).to_frame().T
    overall.insert(0, "candidate", CANDIDATE)
    overall.insert(1, "summary", "average_across_all_activity_signal_rows")

    out_path = CSV_DIR / "realism_average_overall.csv"
    overall.to_csv(out_path, index=False)
    print("Saved:", out_path)

In [52]:
# ============================================================
# Nearest-neighbor distance metrics
# ============================================================

def nearest_mse_distances(
    A: np.ndarray,
    B: np.ndarray,
    batch_size: int = DISTANCE_BATCH_SIZE,
    exclude_self: bool = False,
) -> np.ndarray:
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)

    if A.ndim != 2 or B.ndim != 2:
        raise ValueError("A and B must be 2D arrays.")

    if A.shape[1] != B.shape[1]:
        raise ValueError(f"Feature dimension mismatch: {A.shape[1]} vs {B.shape[1]}")

    n_A = A.shape[0]
    n_B = B.shape[0]
    feature_dim = A.shape[1]

    if n_A == 0 or n_B == 0:
        return np.array([], dtype=np.float32)

    B_norm = np.sum(B * B, axis=1, keepdims=True).T
    nearest = np.empty(n_A, dtype=np.float32)

    same_array = exclude_self and n_A == n_B and np.array_equal(A, B)

    for start in range(0, n_A, batch_size):
        end = min(start + batch_size, n_A)
        A_batch = A[start:end]

        A_norm = np.sum(A_batch * A_batch, axis=1, keepdims=True)
        distances = (A_norm + B_norm - 2.0 * (A_batch @ B.T)) / float(feature_dim)
        distances = np.maximum(distances, 0.0)

        if same_array:
            row_indices = np.arange(end - start)
            col_indices = np.arange(start, end)
            distances[row_indices, col_indices] = np.inf

        nearest[start:end] = np.min(distances, axis=1)

    return nearest


def summarize_mse(values: np.ndarray, prefix: str) -> dict:
    return {
        f"{prefix}_mean": safe_mean(values),
        f"{prefix}_median": safe_percentile(values, 50),
        f"{prefix}_p01": safe_percentile(values, 1),
        f"{prefix}_p05": safe_percentile(values, 5),
        f"{prefix}_p95": safe_percentile(values, 95),
    }


def compute_copy_diversity_table(real_train_X, real_train_y, synthetic_X, synthetic_y):
    rng = np.random.default_rng(RANDOM_SEED)
    rows = []

    for signal_name in SIGNAL_NAMES:
        real_signal = get_signal_windows(real_train_X, signal_name)
        synthetic_signal = get_signal_windows(synthetic_X, signal_name)

        print("\n" + "=" * 70)
        print(f"Copy/diversity metrics | Candidate: {CANDIDATE} | Signal: {signal_name}")
        print("=" * 70)

        for activity_id in ACTIVITY_IDS:
            real_indices = sample_indices_for_activity(
                y=real_train_y,
                activity_id=activity_id,
                max_windows=MAX_REAL_TRAIN_WINDOWS_PER_ACTIVITY_FOR_DISTANCE,
                rng=rng,
            )

            synthetic_indices = sample_indices_for_activity(
                y=synthetic_y,
                activity_id=activity_id,
                max_windows=MAX_SYN_WINDOWS_PER_ACTIVITY_FOR_DISTANCE,
                rng=rng,
            )

            real_sample = real_signal[real_indices].astype(np.float32)
            synthetic_sample = synthetic_signal[synthetic_indices].astype(np.float32)

            if len(real_sample) < 2 or len(synthetic_sample) == 0:
                continue

            real_flat = real_sample.reshape(len(real_sample), -1).astype(np.float32)
            synthetic_flat = synthetic_sample.reshape(len(synthetic_sample), -1).astype(np.float32)

            real_to_real_nn = nearest_mse_distances(
                A=real_flat,
                B=real_flat,
                batch_size=DISTANCE_BATCH_SIZE,
                exclude_self=True,
            )

            synthetic_to_real_nn = nearest_mse_distances(
                A=synthetic_flat,
                B=real_flat,
                batch_size=DISTANCE_BATCH_SIZE,
                exclude_self=False,
            )

            if len(synthetic_flat) >= 2:
                synthetic_to_synthetic_nn = nearest_mse_distances(
                    A=synthetic_flat,
                    B=synthetic_flat,
                    batch_size=DISTANCE_BATCH_SIZE,
                    exclude_self=True,
                )
            else:
                synthetic_to_synthetic_nn = np.array([], dtype=np.float32)

            real_reference_mean = safe_mean(real_to_real_nn)
            synthetic_to_real_mean = safe_mean(synthetic_to_real_nn)
            synthetic_to_synthetic_mean = safe_mean(synthetic_to_synthetic_nn)
            real_reference_p01 = safe_percentile(real_to_real_nn, 1)

            if np.isfinite(real_reference_mean) and real_reference_mean > 1e-12:
                copy_ratio_mean = float(synthetic_to_real_mean / real_reference_mean)
                synthetic_diversity_ratio_mean = float(
                    synthetic_to_synthetic_mean / real_reference_mean
                )
            else:
                copy_ratio_mean = np.nan
                synthetic_diversity_ratio_mean = np.nan

            if np.isfinite(real_reference_p01):
                near_duplicate_rate_p01 = float(
                    np.mean(synthetic_to_real_nn <= real_reference_p01)
                )
            else:
                near_duplicate_rate_p01 = np.nan

            row = {
                "candidate": CANDIDATE,
                "activity": int(activity_id),
                "signal": signal_name,
                "feature_dim": int(real_flat.shape[1]),
                "real_train_sampled_windows": int(len(real_sample)),
                "synthetic_sampled_windows": int(len(synthetic_sample)),
                **summarize_mse(real_to_real_nn, "real_train_to_nearest_real_train_mse"),
                **summarize_mse(synthetic_to_real_nn, "syn_to_nearest_train_real_mse"),
                **summarize_mse(synthetic_to_synthetic_nn, "syn_to_nearest_syn_mse"),
                "copy_ratio_mean": copy_ratio_mean,
                "near_duplicate_threshold_real_train_p01_mse": real_reference_p01,
                "near_duplicate_rate_p01_lower_is_better": near_duplicate_rate_p01,
                "synthetic_diversity_ratio_mean": synthetic_diversity_ratio_mean,
            }

            rows.append(row)

    df = pd.DataFrame(rows)
    out_path = CSV_DIR / "copy_diversity_by_activity_signal.csv"
    df.to_csv(out_path, index=False)
    print("\nSaved:", out_path)

    return df


def save_copy_diversity_summary_tables(copy_df):
    metric_cols = [
        "real_train_to_nearest_real_train_mse_mean",
        "syn_to_nearest_train_real_mse_mean",
        "syn_to_nearest_syn_mse_mean",
        "copy_ratio_mean",
        "near_duplicate_rate_p01_lower_is_better",
        "synthetic_diversity_ratio_mean",
    ]

    avg_by_signal = (
        copy_df
        .groupby("signal", as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    counts_by_signal = (
        copy_df
        .groupby("signal", as_index=False)[
            ["real_train_sampled_windows", "synthetic_sampled_windows"]
        ]
        .sum()
    )

    avg_by_signal = avg_by_signal.merge(counts_by_signal, on="signal", how="left")
    avg_by_signal.insert(0, "candidate", CANDIDATE)

    avg_by_signal = avg_by_signal[
        ["candidate", "signal", "real_train_sampled_windows", "synthetic_sampled_windows"] + metric_cols
    ]

    out_path = CSV_DIR / "copy_diversity_average_by_signal.csv"
    avg_by_signal.to_csv(out_path, index=False)
    print("Saved:", out_path)

    overall = copy_df[metric_cols].mean(numeric_only=True).to_frame().T
    overall.insert(0, "candidate", CANDIDATE)
    overall.insert(1, "summary", "average_across_all_activity_signal_rows")

    out_path = CSV_DIR / "copy_diversity_average_overall.csv"
    overall.to_csv(out_path, index=False)
    print("Saved:", out_path)

In [53]:
# ============================================================
# Data split summary
# ============================================================

def save_data_split_summary(
    real_train_y,
    real_train_subjects,
    real_val_y,
    real_val_subjects,
    real_test_y,
    real_test_subjects,
    synthetic_y,
    synthetic_subjects,
):
    rows = []

    split_items = [
        ("real_train", real_train_y, real_train_subjects),
        ("real_val", real_val_y, real_val_subjects),
        ("real_test", real_test_y, real_test_subjects),
        ("synthetic_all", synthetic_y, synthetic_subjects),
    ]

    for split_name, y, subjects in split_items:
        row = {
            "candidate": CANDIDATE,
            "split": split_name,
            "num_windows": int(len(y)),
            "num_subjects": int(len(np.unique(subjects))),
            "subjects": ",".join(sorted(np.unique(subjects), key=subject_sort_key)),
        }

        for activity_id in ACTIVITY_IDS:
            row[f"activity_{activity_id}_count"] = int(np.sum(y == activity_id))

        rows.append(row)

    df = pd.DataFrame(rows)
    out_path = CSV_DIR / "data_split_summary.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

In [54]:
# ============================================================
# Main execution
# ============================================================

def main():
    print("=" * 80)
    print("Realism / no-copying / diversity evaluation")
    print("=" * 80)
    print("Candidate:", CANDIDATE)

    real_X, real_y, real_subjects, real_metadata = load_real_data()
    synthetic_X, synthetic_y, synthetic_subjects, synthetic_metadata = load_synthetic_data()

    print("\nReal data:", real_X.shape)
    print("Synthetic data:", synthetic_X.shape)
    print("Synthetic directory:", SYN_DIR)
    print("Output directory:", RESULTS_DIR)

    real_train_X, real_train_y, real_train_subjects, real_train_metadata = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        real_metadata,
        TRAIN_SUBJECTS,
    )

    real_val_X, real_val_y, real_val_subjects, real_val_metadata = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        real_metadata,
        VAL_SUBJECTS,
    )

    real_test_X, real_test_y, real_test_subjects, real_test_metadata = filter_by_subjects(
        real_X,
        real_y,
        real_subjects,
        real_metadata,
        TEST_SUBJECTS,
    )

    save_data_split_summary(
        real_train_y=real_train_y,
        real_train_subjects=real_train_subjects,
        real_val_y=real_val_y,
        real_val_subjects=real_val_subjects,
        real_test_y=real_test_y,
        real_test_subjects=real_test_subjects,
        synthetic_y=synthetic_y,
        synthetic_subjects=synthetic_subjects,
    )

    if REALISM_REFERENCE_SPLIT == "train":
        realism_real_X = real_train_X
        realism_real_y = real_train_y
        realism_reference_name = "real_train"
    elif REALISM_REFERENCE_SPLIT == "test":
        realism_real_X = real_test_X
        realism_real_y = real_test_y
        realism_reference_name = "real_test"
    elif REALISM_REFERENCE_SPLIT == "all":
        realism_real_X = real_X
        realism_real_y = real_y
        realism_reference_name = "real_all"
    else:
        raise ValueError("REALISM_REFERENCE_SPLIT must be 'train', 'test', or 'all'.")

    print("\nComputing realism metrics...")
    realism_df = compute_realism_table(
        real_X=realism_real_X,
        real_y=realism_real_y,
        synthetic_X=synthetic_X,
        synthetic_y=synthetic_y,
        reference_name=realism_reference_name,
    )
    save_realism_summary_tables(realism_df)

    print("\nComputing no-copying and diversity metrics...")
    copy_df = compute_copy_diversity_table(
        real_train_X=real_train_X,
        real_train_y=real_train_y,
        synthetic_X=synthetic_X,
        synthetic_y=synthetic_y,
    )
    save_copy_diversity_summary_tables(copy_df)

    print("\nFinished.")
    print("Saved output tables:")
    print(CSV_DIR / "realism_by_activity_signal.csv")
    print(CSV_DIR / "realism_average_by_signal.csv")
    print(CSV_DIR / "realism_average_overall.csv")
    print(CSV_DIR / "copy_diversity_by_activity_signal.csv")
    print(CSV_DIR / "copy_diversity_average_by_signal.csv")
    print(CSV_DIR / "copy_diversity_average_overall.csv")
    print(CSV_DIR / "data_split_summary.csv")


main()

Realism / no-copying / diversity evaluation
Candidate: timevae

Real data: (46907, 512, 6)
Synthetic data: (46907, 512, 6)
Synthetic directory: /home/iailab42/khans1/projects/ir/data/synthetic/timevae
Output directory: /home/iailab42/khans1/projects/ir/results/timevae/realism_diversity
Saved: /home/iailab42/khans1/projects/ir/results/timevae/realism_diversity/csv/data_split_summary.csv

Computing realism metrics...

Realism metrics | Candidate: timevae | Signal: ACC_x

Realism metrics | Candidate: timevae | Signal: ACC_y

Realism metrics | Candidate: timevae | Signal: ACC_z

Realism metrics | Candidate: timevae | Signal: BVP

Realism metrics | Candidate: timevae | Signal: EDA

Realism metrics | Candidate: timevae | Signal: TEMP

Saved: /home/iailab42/khans1/projects/ir/results/timevae/realism_diversity/csv/realism_by_activity_signal.csv
Saved: /home/iailab42/khans1/projects/ir/results/timevae/realism_diversity/csv/realism_average_by_signal.csv
Saved: /home/iailab42/khans1/projects/ir/r